## Before and After Viral Infection: Is the 3D Interaction Intensity of Genes with Significant Splicing Changes Generally Altered?

This notebook compares the average 3D interaction intensity of "genes with significant splicing changes" under two conditions (e.g., infected vs. uninfected), based on rMATS results and two sets of Hi-C data, and tests significance against a random gene set as control.

Analysis workflow (parameterized and reusable):
- Identify all genes with significant splicing changes (rMATS output, merged across event types, filtered by FDR and delta-PSI thresholds)
- Define analysis regions (gene full length ± padding, e.g., 20 kb)
- Extract contact matrices from two Hi-C datasets (cool/mcool) for each region
- Aggregate interaction values across multiple bins to obtain per-gene average interaction intensity (diagonal and upper-triangle mean, ignoring NaN)
- Calculate the difference between two conditions (log2FC)
- Compare against a random (length-matched) gene set to test significance
- Visualize trends and export result tables and figures

Before running, please fill in file paths and thresholds in the "Parameters" cell.

In [ ]:
# ========== Parameters: modify file paths as needed ==========
from pathlib import Path

# rMATS results: provide a list of MATS.JC.txt or MATS.JCEC.txt files for each event type
RMATS_FILES = [
    # Example:
    "/path/to/rmat/output/SE.MATS.JCEC.txt",
    "/path/to/rmat/output/A5SS.MATS.JCEC.txt",
    "/path/to/rmat/output/A3SS.MATS.JCEC.txt",
    "/path/to/rmat/output/MXE.MATS.JCEC.txt",
    "/path/to/rmat/output/RI.MATS.JCEC.txt",
]

# Choose JC or JCEC results: for labeling only; files are read directly from RMATS_FILES
RMATS_LABEL = "JC"  # "JC" or "JCEC"

# Significance thresholds (for rMATS event-level filtering)
FDR_THRESHOLD = 0.05
DELTA_PSI_THRESHOLD = 0.1  # |IncLevelDifference| >= 0.1

# Gene annotation (GTF, GENCODE recommended; supports .gtf or .gtf.gz)
GTF_PATH = "/path/to/ref/Sus_scrofa.Sscrofa11.1.101.chr.gtf"

# Choose gene_name or gene_id for matching (gene_name is preferred for readability)
GTF_GENE_NAME_KEY = "gene_name"  # or "gene_id"

# Hi-C data (cool or mcool) for two conditions (e.g., uninfected vs. infected)
HIC_A_PATH = "/path/to/hic/PI.balanceNoMTY.mcool"  # or .cool
HIC_B_PATH = "/path/to/hic/WT.balanceNoMTY.mcool"

# For .mcool, specify the resolution; for .cool, this parameter can be ignored
HIC_RESOLUTION = 10000  # e.g., 10000, 25000, 50000

# Region padding (gene full length +/- padding)
REGION_PADDING_BP = 20000

# Output directory (result tables and figures)
OUTPUT_DIR = "/path/to/rmat/output/splicing_hic_compare"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Random control: number of quantile bins for length matching and number of repeats
LENGTH_MATCH_Q = 10
RANDOM_SEED = 13

# Statistical test: whether to try using scipy; will fall back automatically if unavailable
PREFER_SCIPY = True

print("Parameters set. Please check RMATS_FILES / GTF_PATH / HIC_* paths before running.")

In [ ]:
# ========== Imports & Utility Functions ==========
import os
import re
import gzip
import math
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Optional dependency: cooler (required for reading cool/mcool)
try:
    import cooler
except Exception as e:
    cooler = None
    print("[Warning] Failed to import cooler. Please install first: pip install cooler")

# Optional dependency: scipy (used for statistical tests; will fall back if unavailable)
try:
    from scipy.stats import mannwhitneyu
except Exception as e:
    mannwhitneyu = None

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="notebook")


def _open_text_maybe_gz(path: str):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")


def load_rmats_significant_genes(rmats_files, fdr_threshold: float, dpsi_threshold: float):
    """Read multiple rMATS event tables (TSV) and merge significant gene lists.
    Required columns: FDR, IncLevelDifference, GeneID or geneSymbol (column names may vary across versions).
    Returns: DataFrame[gene_name, gene_id?] & Set[gene_name]
    """
    if not rmats_files:
        raise ValueError("RMATS_FILES is empty. Please specify rMATS output file list in the parameters cell.")

    frames = []
    for fp in rmats_files:
        if not Path(fp).exists():
            raise FileNotFoundError(f"rMATS file not found: {fp}")
        df = pd.read_csv(fp, sep="\t", dtype=str)
        # Normalize column names to lowercase
        cols = {c.lower(): c for c in df.columns}
        # Get key columns
        fdr_col = cols.get("fdr")
        dpsi_col = cols.get("incleveldifference")
        # Some versions use 'IncLevelDifference' as column name
        if fdr_col is None and "FDR" in df.columns:
            fdr_col = "FDR"
        if dpsi_col is None and "IncLevelDifference" in df.columns:
            dpsi_col = "IncLevelDifference"
        if fdr_col is None or dpsi_col is None:
            raise ValueError(f"rMATS file missing FDR or IncLevelDifference column: {fp}")

        # Gene column: geneSymbol preferred, then GeneID
        gene_col = None
        for cand in ["geneSymbol", "GeneSymbol", "gene_id", "GeneID", "GeneID;" ]:
            if cand in df.columns:
                gene_col = cand
                break
        if gene_col is None:
            # Some rMATS tables have gene info in 'GeneID' or 'geneSymbol'
            # Try case-insensitive matching
            for c in df.columns:
                if c.lower() in ("genesymbol", "geneid"):
                    gene_col = c
                    break
        if gene_col is None:
            raise ValueError(f"rMATS file missing gene column (geneSymbol/GeneID): {fp}")

        # Type conversion
        df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce")
        df[dpsi_col] = pd.to_numeric(df[dpsi_col], errors="coerce")

        sig = df[(df[fdr_col] <= fdr_threshold) & (df[dpsi_col].abs() >= dpsi_threshold)].copy()
        sig.rename(columns={gene_col: "gene"}, inplace=True)
        sig = sig[["gene"]].dropna()
        frames.append(sig)

    sig_genes = pd.concat(frames, ignore_index=True).drop_duplicates()

    # Clean gene names (remove semicolons / handle multi-gene merges)
    def _split_first(g):
        if pd.isna(g):
            return g
        return str(g).split(",")[0].split(";")[0].strip()

    sig_genes["gene"] = sig_genes["gene"].map(_split_first)
    sig_genes = sig_genes[sig_genes["gene"].astype(bool)].drop_duplicates()
    gene_set = set(sig_genes["gene"].tolist())
    return sig_genes, gene_set


def parse_gtf_gene_coords(gtf_path: str, gene_name_key: str = "gene_name"):
    """Parse gene coordinates from GTF (feature == 'gene').
    Returns DataFrame: [chrom, start, end, strand, gene_name, gene_id, gene_length]
    """
    if not Path(gtf_path).exists():
        raise FileNotFoundError(f"GTF file not found: {gtf_path}")

    records = []
    with _open_text_maybe_gz(gtf_path) as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            chrom, source, feature, start, end, score, strand, frame, attrs = parts
            if feature != "gene":
                continue
            start = int(start) - 1  # GTF 1-based -> 0-based
            end = int(end)

            # Parse attributes
            attr_dict = {}
            for kv in re.finditer(r"(\S+)\s+\"([^\"]*)\";", attrs):
                attr_dict[kv.group(1)] = kv.group(2)

            gene_name = attr_dict.get("gene_name") or attr_dict.get("gene") or attr_dict.get("Name")
            gene_id = attr_dict.get("gene_id") or attr_dict.get("ID")

            # Name selection
            name_for_match = (attr_dict.get(gene_name_key) if gene_name_key in ("gene_name", "gene_id") else gene_name) or gene_name or gene_id

            records.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "strand": strand,
                "gene_name": gene_name,
                "gene_id": gene_id,
                "name_for_match": name_for_match,
                "gene_length": max(0, end - start)
            })

    df = pd.DataFrame.from_records(records)
    # Deduplicate: keep the longest span for same-name genes (conservative)
    df.sort_values(["name_for_match", "gene_length"], ascending=[True, False], inplace=True)
    df = df.drop_duplicates(subset=["name_for_match"], keep="first").reset_index(drop=True)
    return df


def add_padding_regions(df_genes: pd.DataFrame, padding_bp: int):
    df = df_genes.copy()
    df["pad_start"] = (df["start"].astype(int) - int(padding_bp)).clip(lower=0)
    df["pad_end"] = df["end"].astype(int) + int(padding_bp)
    return df


def _ensure_cooler(path: str, resolution: int = None):
    if cooler is None:
        raise RuntimeError("The 'cooler' package is required to read Hi-C data: pip install cooler")
    path = str(path)
    if path.endswith(".mcool"):
        if resolution is None:
            raise ValueError("HIC_RESOLUTION must be provided for mcool files")
        uri = f"{path}::/resolutions/{int(resolution)}"
        return cooler.Cooler(uri)
    return cooler.Cooler(path)


def _chrom_name_adapter(cool_obj, chrom: str):
    """Adapt GTF chromosome naming to match cooler conventions (add/remove 'chr' prefix)."""
    names = set(cool_obj.chromnames)
    if chrom in names:
        return chrom
    if chrom.startswith("chr"):
        c = chrom[3:]
        if c in names:
            return c
    else:
        c = "chr" + chrom
        if c in names:
            return c
    return chrom  # Return original as fallback; fetch will raise an exception if it fails


def fetch_region_mean_intensity(cool_obj, chrom: str, start: int, end: int):
    """Extract the contact sub-matrix for a region and compute mean interaction intensity (diagonal and upper triangle), ignoring NaN."""
    try:
        adj_chrom = _chrom_name_adapter(cool_obj, chrom)
        region = f"{adj_chrom}:{int(start)}-{int(end)}"
        mat = cool_obj.matrix(balance=True).fetch(region)
        if mat.size == 0:
            return np.nan
        # Upper triangle (including diagonal)
        triu = np.triu(mat)
        if np.isnan(triu).all():
            return np.nan
        return np.nanmean(triu)
    except Exception:
        return np.nan


def compute_gene_intensity_table(df_regions: pd.DataFrame, cool_a, cool_b):
    rows = []
    for _, r in df_regions.iterrows():
        mean_a = fetch_region_mean_intensity(cool_a, r["chrom"], r["pad_start"], r["pad_end"]) if cool_a else np.nan
        mean_b = fetch_region_mean_intensity(cool_b, r["chrom"], r["pad_start"], r["pad_end"]) if cool_b else np.nan
        rows.append({
            "gene_name": r["gene_name"],
            "name_for_match": r["name_for_match"],
            "chrom": r["chrom"],
            "pad_start": r["pad_start"],
            "pad_end": r["pad_end"],
            "gene_length": r["gene_length"],
            "mean_A": mean_a,
            "mean_B": mean_b,
        })
    df = pd.DataFrame(rows)
    # Compute log2FC (B vs A)
    df["log2FC_B_over_A"] = np.log2((df["mean_B"].astype(float) + 1e-9) / (df["mean_A"].astype(float) + 1e-9))
    return df


def length_matched_random(df_all: pd.DataFrame, df_target: pd.DataFrame, q: int = 10, seed: int = 13):
    """Bin genes by length quantiles and sample the same number of random genes (excluding targets) from each bin."""
    rng = np.random.default_rng(seed)
    df = df_all.copy()
    # Binning
    try:
        df["len_bin"] = pd.qcut(df["gene_length"].rank(method="first"), q=q, labels=False, duplicates="drop")
    except Exception:
        df["len_bin"] = 0
    target = df.merge(df_target[["name_for_match"]].assign(_is_target=1), on="name_for_match", how="left")
    target["_is_target"] = target["_is_target"].fillna(0).astype(int)

    samples = []
    for bin_id, g in target.groupby("len_bin"):
        n_target = (g["_is_target"] == 1).sum()
        if n_target == 0:
            continue
        pool = g[g["_is_target"] == 0]
        if len(pool) == 0:
            continue
        take = min(n_target, len(pool))
        idx = rng.choice(pool.index.values, size=take, replace=False)
        samples.append(pool.loc[idx, ["name_for_match"]])
    if not samples:
        return pd.DataFrame(columns=["name_for_match"])  # empty
    return pd.concat(samples, ignore_index=True)


def try_mannwhitneyu(x, y):
    x = pd.Series(x).dropna().values
    y = pd.Series(y).dropna().values
    if len(x) == 0 or len(y) == 0:
        return np.nan
    if mannwhitneyu is not None:
        res = mannwhitneyu(x, y, alternative="two-sided")
        return res.pvalue
    # Fallback: use simple permutation to approximate mean difference significance
    rng = np.random.default_rng(RANDOM_SEED)
    obs = abs(np.nanmean(x) - np.nanmean(y))
    z = np.concatenate([x, y])
    n = len(x)
    B = 2000 if len(z) < 1000 else 1000
    cnt = 0
    for _ in range(B):
        rng.shuffle(z)
        x2 = z[:n]
        y2 = z[n:]
        if abs(np.nanmean(x2) - np.nanmean(y2)) >= obs:
            cnt += 1
    return (cnt + 1) / (B + 1)

In [ ]:
# ========== Data Preparation: Load rMATS Significant Genes + GTF Gene Coordinates ==========
# 1) rMATS significant genes
sig_df, sig_set = load_rmats_significant_genes(RMATS_FILES, FDR_THRESHOLD, DELTA_PSI_THRESHOLD)
print(f"Number of rMATS significant genes: {len(sig_set)}")

# 2) GTF gene coordinates
ann_df = parse_gtf_gene_coords(GTF_PATH, gene_name_key=GTF_GENE_NAME_KEY)
print(f"GTF gene entries: {len(ann_df)}")

# 3) Map significant genes to annotation coordinates (match by gene_name by default; use gene_id if configured)
key = "name_for_match"
# Build matching column in sig_df (unified with GTF's name_for_match)
sig_df = sig_df.rename(columns={"gene": key})

merged_sig = ann_df.merge(sig_df[[key]].drop_duplicates(), on=key, how="inner")
print(f"Significant genes successfully matched to coordinates: {len(merged_sig)}")

# 4) Generate padded regions
sig_regions = add_padding_regions(merged_sig, REGION_PADDING_BP)
print(sig_regions.head(3))

# 5) Also prepare the full gene set for random control (optional filter: keep only autosomes)
all_genes = add_padding_regions(ann_df, REGION_PADDING_BP)

# Simple filter example: keep chr1-22, X, Y (will be re-mapped when adapting to cooler chromosome names)
_keep = ann_df["chrom"].astype(str).str.replace("chr", "", regex=False)
mask_chr = _keep.isin([str(i) for i in range(1, 23)] + ["X", "Y"]) | ann_df["chrom"].astype(str).str.startswith("chr")
sig_regions = sig_regions[mask_chr.loc[sig_regions.index]].reset_index(drop=True)
all_genes = all_genes[mask_chr].reset_index(drop=True)
print(f"After filtering: {len(sig_regions)} significant genes, {len(all_genes)} total genes")

In [ ]:
# ========== Extract Hi-C Interactions and Compute Per-Gene Mean Intensity + log2FC ==========
# Open Hi-C (use specified resolution if mcool)
if HIC_A_PATH and HIC_B_PATH:
    cool_A = _ensure_cooler(HIC_A_PATH, HIC_RESOLUTION)
    cool_B = _ensure_cooler(HIC_B_PATH, HIC_RESOLUTION)
else:
    cool_A = None
    cool_B = None

print("Computing mean interaction intensity for gene regions (may be slow)...")
res_sig = compute_gene_intensity_table(sig_regions, cool_A, cool_B)
print("Done: significant gene intensity table", res_sig.shape)

# Save detailed results
res_sig_path = str(Path(OUTPUT_DIR) / f"sig_genes_contact_{RMATS_LABEL}_res{HIC_RESOLUTION}_pad{REGION_PADDING_BP}.tsv")
res_sig.to_csv(res_sig_path, sep="\t", index=False)
print("Saved:", res_sig_path)

In [ ]:
# ========== Random Length-Matched Control & Significance Test ==========
# Perform length-binned matched sampling based on all genes and significant genes
rand_names = length_matched_random(all_genes[["name_for_match", "gene_length"]],
                                   sig_regions[["name_for_match"]],
                                   q=LENGTH_MATCH_Q,
                                   seed=RANDOM_SEED)
print(f"Random length-matched sample size: {len(rand_names)} (expected ~ number of significant genes)")

# Extract intensity for random genes (compute or use saved results)
all_regions_for_rand = all_genes.merge(rand_names, on="name_for_match", how="inner")
res_rand = compute_gene_intensity_table(all_regions_for_rand, cool_A, cool_B)
print("Done: random gene intensity table", res_rand.shape)

# Statistical test: compare log2FC distributions between the two groups
pval = try_mannwhitneyu(res_sig["log2FC_B_over_A"], res_rand["log2FC_B_over_A"])
print(f"Mann-Whitney U (or permutation approximation) p-value: {pval}")

# Save random control and summary
res_rand_path = str(Path(OUTPUT_DIR) / f"random_matched_contact_{RMATS_LABEL}_res{HIC_RESOLUTION}_pad{REGION_PADDING_BP}.tsv")
res_rand.to_csv(res_rand_path, sep="\t", index=False)
print("Saved:", res_rand_path)

summary = {
    "n_sig_genes": int(len(res_sig)),
    "n_rand_genes": int(len(res_rand)),
    "median_log2FC_sig": float(pd.to_numeric(res_sig["log2FC_B_over_A"], errors="coerce").median(skipna=True)),
    "median_log2FC_rand": float(pd.to_numeric(res_rand["log2FC_B_over_A"], errors="coerce").median(skipna=True)),
    "pvalue": float(pval) if (pval is not None and not pd.isna(pval)) else None,
}
with open(Path(OUTPUT_DIR) / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary:", summary)

In [ ]:
# ========== Visualization ==========
fig1_path = str(Path(OUTPUT_DIR) / f"log2fc_box_violin_{RMATS_LABEL}_res{HIC_RESOLUTION}_pad{REGION_PADDING_BP}.pdf")
fig2_path = str(Path(OUTPUT_DIR) / f"log2fc_ecdf_{RMATS_LABEL}_res{HIC_RESOLUTION}_pad{REGION_PADDING_BP}.pdf")

# Infer condition labels from Hi-C paths for plot title display: B/A = {B}/{A}
from pathlib import Path as _P

def _infer_label(p):
    try:
        name = _P(p).name
        # Prefer the first segment split by dot, then by underscore
        if "." in name:
            return name.split(".")[0]
        if "_" in name:
            return name.split("_")[0]
        return name
    except Exception:
        return str(p)

A_LABEL = _infer_label(HIC_A_PATH)
B_LABEL = _infer_label(HIC_B_PATH)
ratio_str = f"{B_LABEL}/{A_LABEL}"

plot_df = pd.concat([
    pd.DataFrame({"group": "Splicing-significant", "log2FC": res_sig["log2FC_B_over_A"]}),
    pd.DataFrame({"group": "Random-matched", "log2FC": res_rand["log2FC_B_over_A"]}),
], ignore_index=True)

plt.figure(figsize=(6.5, 4.5))
ax = sns.violinplot(data=plot_df, x="group", y="log2FC", inner=None, cut=0)
sns.boxplot(data=plot_df, x="group", y="log2FC", showcaps=True, boxprops={"facecolor":"white"},
            showfliers=False, whiskerprops={"linewidth":1}, width=0.25)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)

# Median annotation (black diamond)
med_sig = np.nanmedian(res_sig["log2FC_B_over_A"]) if len(res_sig) else np.nan
med_rand = np.nanmedian(res_rand["log2FC_B_over_A"]) if len(res_rand) else np.nan
plt.scatter([0, 1], [med_sig, med_rand], color="black", s=24, marker="D", zorder=4)

# Significance annotation (Mann-Whitney U): draw a horizontal bar with asterisks above both groups
y_min = np.nanmin(plot_df["log2FC"]) if plot_df["log2FC"].notna().any() else 0.0
y_max = np.nanmax(plot_df["log2FC"]) if plot_df["log2FC"].notna().any() else 1.0
span = (y_max - y_min) if y_max > y_min else 1.0
x1, x2 = 0, 1
h = 0.02 * span
y_bar = y_max + 0.05 * span
plt.plot([x1, x1, x2, x2], [y_bar, y_bar + h, y_bar + h, y_bar], color="black", linewidth=1)

def _p_to_stars(p):
    if p is None or (isinstance(p, float) and math.isnan(p)):
        return "n.s."
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "n.s."

stars = _p_to_stars(pval)
plt.text((x1 + x2) / 2, y_bar + h * 1.5, f"{stars} (p={pval:.2e})", ha="center", va="bottom", fontsize=10)
# Extend y-axis to accommodate annotations
cur_ylim = plt.ylim()
plt.ylim(cur_ylim[0], max(cur_ylim[1], y_bar + h * 3))

plt.ylabel(f"log2FC({ratio_str})")
plt.title(f"log2FC({ratio_str}) of contact intensity")
plt.tight_layout()
plt.savefig(fig1_path, dpi=180)
plt.show()
print("Saved:", fig1_path)

plt.figure(figsize=(6.5, 4.5))
sns.ecdfplot(data=plot_df, x="log2FC", hue="group")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(f"log2FC({ratio_str})")
plt.title(f"ECDF of log2FC({ratio_str})")
plt.tight_layout()
plt.savefig(fig2_path, dpi=180)
plt.show()
print("Saved:", fig2_path)